In [1]:
import pandas as pd 
from pathlib import Path
from openpyxl import load_workbook
from openpyxl.styles import PatternFill


In [2]:
input_file = Path('/Users/jeremypalmerio/Desktop/Python/ESIS/TEST_EXPORT_ESIS.csv')
suivis_file = Path('/Users/jeremypalmerio/Desktop/Python/ESIS/suivis patients aller vers (2).xlsx')

In [3]:
year = 2025
suivis_excel = load_workbook(suivis_file)
suivis_df = pd.read_excel(suivis_file, sheet_name=str(year))

/opt/miniconda3/envs/py3/lib/python3.10/site-packages/openpyxl/reader/excel.py:237: UserWarning: Conditional Formatting extension is not supported and will be removed
  ws_parser.bind_all()
/opt/miniconda3/envs/py3/lib/python3.10/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Conditional Formatting extension is not supported and will be removed
  for idx, row in parser.parse():


In [4]:
suivis_df.head()

,date,dep,lieu de l'action,typologie de l'action,titre,nom de naissance,prénom,date de naissance,âge (calcul automatique),n° tel entrer les chiffres sans espace (mise en forme spécifique),...,délais en mois jusqu'au frottis suivant,si frottis + suivi,type de suivi,situation finale DOCCU,mail personne référente (suivi +),nom du labo,contact labo,date saisie,date dernière vérification,remarques
0,2025-03-17,75.0,Air Liquide,Mars bleu,Mr,PALMERIO,Jer,1973-10-25,51.0,123451234.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-03-17,75.0,Air Liquide,Mars bleu,Mme,CELIKTIN,Ada,1973-11-17,51.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Get SSN, test date, test result, and last test date from E SIS export


In [5]:
input_df = pd.read_csv(input_file, delimiter=';')
input_df.head()

,Identifiant,Identifiant2,id_bci,NSS,clé NSS,civilite,nom,nom_jf,prenom,adressetot,...,localite,ddn,num_portable,date_invitation,date_prescrisption,type_invitation,date_Test_Avant_Invitation,date_Realisation_test,resultat_test,Utilisateur
0,29594906,29594906,"9,11E+12",2 73 10 80 000 000,57,Madame,PALMERIO,CELIKTIN,ADA,12 Rue DES AIGRE,...,PALAISEAU,25/10/1973,NaN,06/11/2023,01/11/2025,Spécifique,NaN,NaN,NaN,SM
1,29593462,29593462,"9,11E+12",2 73 11 80 000 000,8,Monsieur,PALMERIO,PALMERIO,JER,9 Rue CHARLES,...,PALAISEAU,17/11/1973,123451234.0,04/12/2023,21/11/2025,Spécifique,29/11/2020,29/11/2025,Négatif,SM


In [6]:
input_df.columns

Index(['Identifiant', 'Identifiant2', 'id_bci', 'NSS', 'clé NSS', 'civilite',
       'nom', 'nom_jf', 'prenom', 'adressetot', 'adresse_2', 'cp', 'localite',
       'ddn', 'num_portable', 'date_invitation', 'date_prescrisption',
       'type_invitation', 'date_Test_Avant_Invitation',
       'date_Realisation_test', 'resultat_test', 'Utilisateur'],
      dtype='object')

In [7]:
def clean_result(result):
    result_map = {'Négatif': '-', 'Postif': '+', '': 'N.A.'}
    return result_map.get(result, 'N.A.')

def clean_date(date):
    if pd.isna(date):
        return None
    parts = date.split('/')
    return f"{parts[1]}/{parts[0]}/{parts[2]}"

def get_age(ddn, year):
    if pd.isna(ddn):
        return None
    age = year - pd.to_datetime(ddn).year
    return age

def get_rang(age):
    if age is None:
        return None
    rang = (age - 50)// 2 + 1
    return f'R{rang}'

def get_patient_data(input_df):
    """
    Extract nss, test date, result and previous test data from the E SIS export DataFrame. 
    Returns a dictionary with nss as keys and a dictionary of test data as values. 
    Entries with missing data are filled with nan.
    """
    result_map = {'Négatif': '-', 'Postif': '+', '': 'N.A.'}
    patient_data = {}
    for index, row in input_df.iterrows():
        nss = int(row['NSS'].replace(' ', '')) 
        test_date = clean_date(row['date_Realisation_test'])
        test_result = clean_result(row['resultat_test'])
        last_test_date = clean_date(row['date_Test_Avant_Invitation'])
        age = get_age(row['ddn'], 2025-1)
        rang = get_rang(age)

        patient_data[nss] = {'age': age, 'test_date': test_date, 'test_result': test_result, 'last_test_date': last_test_date, 'rang': rang}
    return patient_data

In [8]:
get_rang(55)

'R3'

In [9]:
patient_data = get_patient_data(input_df)
patient_data

/var/folders/8w/zsk2cpvx41509nlpl3s_j5lc0000gn/T/ipykernel_10884/2690718566.py:14: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  age = year - pd.to_datetime(ddn).year


{2731080000000: {'age': 51,
  'test_date': None,
  'test_result': 'N.A.',
  'last_test_date': None,
  'rang': 'R1'},
 2731180000000: {'age': 51,
  'test_date': '11/29/2025',
  'test_result': '-',
  'last_test_date': '11/29/2020',
  'rang': 'R1'}}

In [10]:
rows = suivis_df.loc[suivis_df['immatriculation sécu entrer les chiffres sans espace (mise en forme spécifique)'] == 2731180000000]

In [11]:
# columns to update: (1, 11) = ssn, (1, 27) = test date, (1, 28) = test result, (1, 32) = last test date

In [ ]:
def fill_patient(suivis_excel, suivis_df, patient_data):
    sheet = suivis_excel['2025']

    for patient_nss, data in patient_data.items():
        row = suivis_df.loc[suivis_df['immatriculation sécu entrer les chiffres sans espace (mise en forme spécifique)'] == patient_nss]
        if row.empty:
            print(f"Patient with NSS {patient_nss} not found in suivis_df.")
            return None
        elif len(row) > 1: 
            print(f"Multiple entries found for NSS {patient_nss}. Please check the data.")

        row_index = row.index.values[0] + 2 # +2 because of header and 0-indexing

        # change cells
        sheet.cell(row=row_index, column=27).value = data['test_date']
        sheet.cell(row=row_index, column=28).value = data['test_result'] 
        sheet.cell(row=row_index, column=32).value = data['last_test_date']
        sheet.cell(row=row_index, column=31).value = data['rang']

        none_color = PatternFill(start_color="FF215F9A", end_color="FF215F9A", fill_type="solid")
        else_color = PatternFill(start_color="FFA6CAEC", end_color="FFA6CAEC", fill_type="solid")
       
        if data['test_date'] is None:
            sheet.cell(row=row_index, column=27).fill = none_color
        else:
            sheet.cell(row=row_index, column=27).fill = else_color
        
        if data['last_test_date'] is None: 
            sheet.cell(row=row_index, column=32).fill = none_color 
        else: 
            sheet.cell(row=row_index, column=32).fill = else_color


    return suivis_excel

In [13]:
fill_patient(suivis_excel=suivis_excel, suivis_df=suivis_df, patient_data=patient_data)